# Módulo 37 · RDKit II: subestructuras, edición y similitud

Cuaderno del curso *Aprende Python desde cero*. Ejecuta las celdas en orden (**Entorno de ejecución → Ejecutar todo**, o Shift+Enter celda a celda). Las explicaciones completas están en el módulo 37 de la web.

In [ ]:
# Instalación (solo la primera vez en Colab; en tu ordenador: pip install rdkit seaborn pandas matplotlib)
!pip -q install rdkit seaborn


Archivos de datos del curso (se crean en la carpeta de trabajo):

In [ ]:
# Archivos de datos del curso

print("Archivos creados:", [])

> **RDKit se ejecuta fuera del navegador.** No existe una versión de RDKit para Python compilada para la web (ni siquiera en Pyodide), así que en este módulo el código se muestra para ejecutarlo en **Google Colab** o en tu ordenador. Abre `notebooks/modulo-37-rdkit-2.ipynb` (viene en la carpeta del curso) en [colab.research.google.com](https://colab.research.google.com) con *Archivo → Subir cuaderno*: incluye la instalación, los archivos de datos y la comprobación de cada ejercicio.

Una vez tienes moléculas como objetos, puedes preguntarles cosas: ¿contienen un grupo funcional?, ¿qué pasa si quito este átomo?, ¿cuánto se parecen a otra molécula? Estas tres operaciones están en el corazón del diseño de fármacos.

### Buscar subestructuras con SMARTS

In [ ]:
from rdkit import Chem

paracetamol = Chem.MolFromSmiles("CC(=O)NC1=CC=C(C=C1)O")
patrones = {"carbono aromático": "c", "hidroxilo": "[OX2H]", "amida": "C(=O)N",
            "anillo bencénico": "c1ccccc1", "ácido carboxílico": "C(=O)[OH]"}
for nombre, smarts in patrones.items():
    patron = Chem.MolFromSmarts(smarts)
    coincidencias = paracetamol.GetSubstructMatches(patron)
    print(f"{nombre:18s} {len(coincidencias)} coincidencia(s): {coincidencias}")

print(paracetamol.HasSubstructMatch(Chem.MolFromSmarts("C(=O)N")))

> **Error habitual.** En uno de los scripts de clase aparece `mol_paracetamol.c(substructura_ciclica)`: ese método no existe y da `AttributeError`. El correcto es `GetSubstructMatches`. Cuando no recuerdes un nombre, usa `dir(objeto)` (módulo 9).

In [ ]:
from rdkit.Chem import Draw
amida = Chem.MolFromSmarts("C(=O)N")
atomos = paracetamol.GetSubstructMatch(amida)
from rdkit.Chem.Draw import rdMolDraw2D
d = rdMolDraw2D.MolDraw2DSVG(320, 220)
d.DrawMolecule(paracetamol, highlightAtoms=atomos, legend="Amida resaltada")
d.FinishDrawing()
from IPython.display import SVG
SVG(d.GetDrawingText())

### Modificar moléculas

In [ ]:
from rdkit import Chem

mol = Chem.MolFromSmiles("CC(=O)NC1=CC=C(C=C1)O")
for atomo in mol.GetAtoms():
    print(atomo.GetIdx(), atomo.GetSymbol(), end=" | ")
print()

editable = Chem.RWMol(mol)                 # versión modificable
editable.RemoveAtom(2)                     # quitamos el oxígeno del carbonilo (índice 2)
nueva = editable.GetMol()
Chem.SanitizeMol(nueva)
print("Sin el O del carbonilo:", Chem.MolToSmiles(nueva))

con_h = Chem.AddHs(mol)
print("Átomos sin H:", mol.GetNumAtoms(), "| con H explícitos:", con_h.GetNumAtoms())
print("Quitando H de nuevo:", Chem.RemoveHs(con_h).GetNumAtoms())
nueva    # se dibuja al final de la celda

> **Consejo.** Los índices de los átomos cambian al eliminar átomos. Si vas a quitar varios, hazlo de mayor a menor índice.

### Huellas moleculares y similitud de Tanimoto

Para comparar moléculas, cada una se convierte en una **huella** (fingerprint): un vector de bits donde cada bit indica la presencia de un fragmento. La huella de **Morgan** (radio 2 ≈ ECFP4) codifica el entorno de cada átomo hasta 2 enlaces. El **coeficiente de Tanimoto** mide bits compartidos / bits totales: 1 = idénticas, 0 = nada en común.

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

generador = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
ibuprofeno = Chem.MolFromSmiles("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O")
fp = generador.GetFingerprint(ibuprofeno)
print("Bits encendidos:", fp.GetNumOnBits(), "de", fp.GetNumBits())
print(fp.ToBitString()[:80], "...")

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

farmacos = {"Aspirina": "CC(=O)OC1=CC=CC=C1C(=O)O", "Paracetamol": "CC(=O)NC1=CC=C(C=C1)O",
            "Ibuprofeno": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "Naproxeno": "COC1=CC2=CC(=CC=C2C=C1)C(C)C(=O)O",
            "Diazepam": "CN1C(=O)CN=C(C2=C1C=CC(=C2)Cl)C3=CC=CC=C3", "Cafeína": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"}
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in farmacos.values()]
matriz = [[DataStructs.TanimotoSimilarity(a, b) for b in fps] for a in fps]
df = pd.DataFrame(matriz, index=farmacos, columns=farmacos).round(2)
sns.heatmap(df, annot=True, cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Similitud de Tanimoto (Morgan r=2)")
plt.show()

> **Nota.** Fíjate en que ibuprofeno y naproxeno (dos ácidos arilpropiónicos) son los más parecidos. Así funciona el **cribado por similitud**: buscar en bases de millones de compuestos los que más se parecen a un fármaco conocido.

In [ ]:
from rdkit.Chem import AllChem
fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)   # API antigua (obsoleta)

*Basado en clase_9: rdkit_4_modificar_moleculas.py rdkit_6_GetSubstructMatches.py rdkit_7_Morgan.Tanimoto.py Script7.py*

#### ✏️ Ejercicio: Buscador de análogos

Escribe `mas_similares(consulta, biblioteca, n)` que devuelva una lista con los `n` nombres de `biblioteca` (diccionario nombre→SMILES) más parecidos a la molécula `consulta` (SMILES), ordenados de mayor a menor Tanimoto (Morgan radio 2, 2048 bits).

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

def mas_similares(consulta, biblioteca, n=2):
    pass

biblioteca = {"Aspirina": "CC(=O)OC1=CC=CC=C1C(=O)O", "Paracetamol": "CC(=O)NC1=CC=C(C=C1)O",
              "Naproxeno": "COC1=CC2=CC(=CC=C2C=C1)C(C)C(=O)O", "Cafeína": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
              "Ketoprofeno": "CC(C1=CC(=CC=C1)C(=O)C2=CC=CC=C2)C(=O)O"}
print(mas_similares("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", biblioteca))   # ibuprofeno

Cuando termines, ejecuta la celda de comprobación (no la modifiques):

In [ ]:
# Comprobación automática
r = mas_similares("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", biblioteca, 2)
assert isinstance(r, list) and len(r) == 2, f"Debe devolver una lista de 2 nombres; devuelve {r}"
assert set(r) == {"Naproxeno", "Ketoprofeno"}, f"Los análogos más cercanos del ibuprofeno son Naproxeno y Ketoprofeno; tienes {r}"
print("✅ ¡Correcto!")

<details><summary>Ver solución explicada</summary>

Calcula la huella de la consulta una vez; para cada elemento de la biblioteca calcula su similitud y guarda pares (nombre, similitud). Ordena con `sorted(..., key=lambda x: x[1], reverse=True)` y quédate con los nombres de los primeros n.

```python
def mas_similares(consulta, biblioteca, n=2):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fp_q = gen.GetFingerprint(Chem.MolFromSmiles(consulta))
    puntuaciones = [(nombre, DataStructs.TanimotoSimilarity(fp_q, gen.GetFingerprint(Chem.MolFromSmiles(smi))))
                    for nombre, smi in biblioteca.items()]
    return [nombre for nombre, _ in sorted(puntuaciones, key=lambda x: x[1], reverse=True)[:n]]
```

</details>

#### Resumen del módulo
`Chem.MolFromSmarts` + `GetSubstructMatches` / `HasSubstructMatch` buscan grupos funcionales.`Chem.RWMol` permite añadir/quitar átomos y enlaces; `AddHs`/`RemoveHs` gestionan hidrógenos.Huella de Morgan: `rdFingerprintGenerator.GetMorganGenerator(radius=2)`.`DataStructs.TanimotoSimilarity`: similitud entre 0 y 1.